In [2]:
# Cell 1: Import libraries and load both datasets
from datasets import load_dataset
import pandas as pd

# Load sentiment dataset (Twitter sentiment: positive/negative/neutral)
sentiment_data = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Load intent dataset (banking customer queries: 77 intents)
# Using the mteb mirror since it's already in Parquet format
intent_data = load_dataset("mteb/banking77")

print("Sentiment dataset loaded:", sentiment_data)
print("\nIntent dataset loaded:", intent_data)

d:\Projects\sentiment-intent-pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentiment dataset loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Intent dataset loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})


In [3]:
# Cell 2: Look at the actual data
sentiment_df = sentiment_data['train'].to_pandas()
intent_df = intent_data['train'].to_pandas()

print("SENTIMENT DATA SAMPLE:")
print(sentiment_df.head())
print("\nSentiment label distribution:")
print(sentiment_df['label'].value_counts())

print("\n\nINTENT DATA SAMPLE:")
print(intent_df.head())
print("\nNumber of unique intents:", intent_df['label'].nunique())
print("\nSample intent names:")
print(intent_df['label_text'].unique()[:10])

SENTIMENT DATA SAMPLE:
                                                text  label
0  "QT @user In the original draft of the 7th boo...      2
1  "Ben Smith / Smith (concussion) remains out of...      1
2  Sorry bout the stream last night I crashed out...      1
3  Chase Headley's RBI double in the 8th inning o...      1
4  @user Alciato: Bee will invest 150 million in ...      2

Sentiment label distribution:
label
1    20673
2    17849
0     7093
Name: count, dtype: int64


INTENT DATA SAMPLE:
                                                text  label    label_text
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival
3  Can I track my card while it is in the process...     11  card_arrival
4  How do I know if I will get my card, or if it ...     11  card_arrival

Number of unique intents: 77

Sample intent names:
<Ar

In [4]:
# Cell 3: One-time downloads for text processing
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gokul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Gokul\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Gokul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# Cell 4: Text cleaning function
import re
import spacy
from nltk.corpus import stopwords

# Load spaCy model once (reused for every row — loading it repeatedly is slow)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])  # disable unused parts for speed
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)         # remove URLs
    text = re.sub(r'@\w+', '', text)                   # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)               # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()           # remove extra whitespace

    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text not in stop_words and len(token.text) > 1]
    return ' '.join(tokens)

# Quick test on one example
sample = sentiment_df['text'].iloc[0]
print("BEFORE:", sample)
print("AFTER:", clean_text(sample))

BEFORE: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
AFTER: qt original draft th book remus lupin survive battle hogwart happybirthdayremuslupin


In [6]:
# Cell 5: Apply cleaning to full sentiment dataset (this will take a few minutes)
from tqdm import tqdm
tqdm.pandas()

sentiment_df['clean_text'] = sentiment_df['text'].progress_apply(clean_text)

print(sentiment_df[['text', 'clean_text']].head())

100%|██████████| 45615/45615 [09:02<00:00, 84.01it/s] 


                                                text  \
0  "QT @user In the original draft of the 7th boo...   
1  "Ben Smith / Smith (concussion) remains out of...   
2  Sorry bout the stream last night I crashed out...   
3  Chase Headley's RBI double in the 8th inning o...   
4  @user Alciato: Bee will invest 150 million in ...   

                                          clean_text  
0  qt original draft th book remus lupin survive ...  
1  ben smith smith concussion remain lineup thurs...  
2  sorry bout stream last night crash tonight sur...  
3  chase headley rbi double th inning david price...  
4  alciato bee invest million january another sum...  


In [7]:
# Cell 6: Apply cleaning to intent dataset
intent_df['clean_text'] = intent_df['text'].progress_apply(clean_text)
print(intent_df[['text', 'clean_text']].head())

100%|██████████| 9993/9993 [01:46<00:00, 93.82it/s] 

                                                text  \
0                     I am still waiting on my card?   
1  What can I do if my card still hasn't arrived ...   
2  I have been waiting over a week. Is the card s...   
3  Can I track my card while it is in the process...   
4  How do I know if I will get my card, or if it ...   

                    clean_text  
0              still wait card  
1   card still not arrive week  
2    wait week card still come  
3  track card process delivery  
4           know get card lose  


In [8]:
# Cell 7: Save cleaned datasets to the data/ folder
sentiment_df.to_csv('../data/sentiment_cleaned.csv', index=False)
intent_df.to_csv('../data/intent_cleaned.csv', index=False)

print("Saved successfully.")

Saved successfully.


In [9]:
# Cell 8: Train/test split + TF-IDF + Logistic Regression baseline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Split cleaned data
X = sentiment_df['clean_text']
y = sentiment_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train baseline model (class_weight='balanced' helps with your imbalanced classes)
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

              precision    recall  f1-score   support

    negative       0.44      0.64      0.52      1419
     neutral       0.67      0.59      0.63      4134
    positive       0.69      0.65      0.67      3570

    accuracy                           0.62      9123
   macro avg       0.60      0.63      0.61      9123
weighted avg       0.64      0.62      0.63      9123



In [10]:
# Cell 9: Save the sentiment model and vectorizer
import joblib

joblib.dump(model, '../models/sentiment_model.pkl')
joblib.dump(vectorizer, '../models/sentiment_vectorizer.pkl')

print("Sentiment model saved.")

Sentiment model saved.


In [11]:
# Cell 10: Train/test split + TF-IDF + Logistic Regression for intent
X_intent = intent_df['clean_text']
y_intent = intent_df['label']

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_intent, y_intent, test_size=0.2, random_state=42, stratify=y_intent
)

vectorizer_intent = TfidfVectorizer(max_features=5000)
X_train_i_tfidf = vectorizer_intent.fit_transform(X_train_i)
X_test_i_tfidf = vectorizer_intent.transform(X_test_i)

model_intent = LogisticRegression(max_iter=1000, class_weight='balanced')
model_intent.fit(X_train_i_tfidf, y_train_i)

y_pred_i = model_intent.predict(X_test_i_tfidf)
print(classification_report(y_test_i, y_pred_i))

              precision    recall  f1-score   support

           0       1.00      0.88      0.93        32
           1       1.00      0.95      0.98        22
           2       0.93      1.00      0.96        25
           3       0.89      1.00      0.94        17
           4       1.00      0.88      0.94        25
           5       0.81      0.76      0.79        34
           6       0.90      0.97      0.93        36
           7       0.89      0.81      0.85        31
           8       0.88      0.97      0.92        31
           9       1.00      0.92      0.96        26
          10       0.91      0.83      0.87        12
          11       0.81      0.94      0.87        31
          12       0.86      0.82      0.84        22
          13       0.96      0.93      0.95        28
          14       0.62      0.73      0.67        22
          15       0.84      0.84      0.84        38
          16       0.77      0.88      0.82        34
          17       0.96    

In [12]:
# Cell 11: Save the intent model and vectorizer
joblib.dump(model_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_intent, '../models/intent_vectorizer.pkl')

print("Intent model saved.")

Intent model saved.


In [13]:
# Cell 12: Sanity test both models on made-up examples
test_sentences = [
    "This product is absolutely amazing, I love it!",
    "Terrible service, I want a refund immediately.",
    "It's okay, nothing special but not bad either.",
    "My card hasn't arrived yet, it's been two weeks.",
    "Can you tell me your working hours?"
]

for text in test_sentences:
    cleaned = clean_text(text)

    sent_vec = vectorizer.transform([cleaned])
    sentiment_pred = model.predict(sent_vec)[0]
    sentiment_label = ['negative', 'neutral', 'positive'][sentiment_pred]

    intent_vec = vectorizer_intent.transform([cleaned])
    intent_pred = model_intent.predict(intent_vec)[0]
    intent_label = intent_df[intent_df['label'] == intent_pred]['label_text'].iloc[0]

    print(f"Text: {text}")
    print(f"  → Sentiment: {sentiment_label} | Intent: {intent_label}\n")

Text: This product is absolutely amazing, I love it!
  → Sentiment: positive | Intent: request_refund

Text: Terrible service, I want a refund immediately.
  → Sentiment: negative | Intent: request_refund

Text: It's okay, nothing special but not bad either.
  → Sentiment: negative | Intent: receiving_money

Text: My card hasn't arrived yet, it's been two weeks.
  → Sentiment: negative | Intent: card_arrival

Text: Can you tell me your working hours?
  → Sentiment: neutral | Intent: pending_top_up



In [14]:
# Cell 13: Save intent label map so pipeline.py can use it independently
import json

intent_label_map = dict(zip(intent_df['label'], intent_df['label_text']))
intent_label_map = {int(k): v for k, v in intent_label_map.items()}  # ensure keys are plain ints

with open('../models/intent_label_map.json', 'w') as f:
    json.dump(intent_label_map, f)

print("Saved. Total intents:", len(intent_label_map))

Saved. Total intents: 77


In [15]:
# Cell: Load general customer support intent dataset (Bitext)
from datasets import load_dataset

general_data = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
general_df = general_data['train'].to_pandas()

print(general_df.shape)
print(general_df.columns.tolist())
print(general_df[['instruction', 'intent']].head())

(26872, 5)
['flags', 'instruction', 'category', 'intent', 'response']
                                         instruction        intent
0   question about cancelling order {{Order Number}}  cancel_order
1  i have a question about cancelling oorder {{Or...  cancel_order
2    i need help cancelling puchase {{Order Number}}  cancel_order
3         I need to cancel purchase {{Order Number}}  cancel_order
4  I cannot afford this order, cancel purchase {{...  cancel_order


In [16]:
# Cell: Prepare both datasets with matching column names
banking_subset = intent_df[['text', 'label_text']].rename(columns={'label_text': 'intent_label'})
banking_subset['text'] = banking_subset['text']

general_subset = general_df[['instruction', 'intent']].rename(columns={'instruction': 'text', 'intent': 'intent_label'})

combined_df = pd.concat([banking_subset, general_subset], ignore_index=True)
combined_df = combined_df.dropna(subset=['text', 'intent_label'])

print("Combined shape:", combined_df.shape)
print("Total unique intents:", combined_df['intent_label'].nunique())
print(combined_df['intent_label'].value_counts().tail(10))

Combined shape: (36865, 2)
Total unique intents: 104
intent_label
top_up_limits                  97
get_disposable_virtual_card    97
receiving_money                95
compromised_card               86
atm_support                    86
lost_or_stolen_card            82
card_swallowed                 61
card_acceptance                59
virtual_card_not_working       41
contactless_not_working        35
Name: count, dtype: int64


In [17]:
# Cell: Clean combined dataset (reuses your existing clean_text function)
from tqdm import tqdm
tqdm.pandas()

combined_df['clean_text'] = combined_df['text'].progress_apply(clean_text)

100%|██████████| 36865/36865 [05:38<00:00, 108.81it/s]


In [18]:
# Cell: Save combined dataset
combined_df.to_csv('../data/combined_intent_cleaned.csv', index=False)
print("Saved.")

Saved.


In [19]:
# Cell: Train/test split + TF-IDF + Logistic Regression on combined intents
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
combined_df['label_encoded'] = le.fit_transform(combined_df['intent_label'])

X_c = combined_df['clean_text']
y_c = combined_df['label_encoded']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

vectorizer_combined = TfidfVectorizer(max_features=8000)
X_train_c_tfidf = vectorizer_combined.fit_transform(X_train_c)
X_test_c_tfidf = vectorizer_combined.transform(X_test_c)

model_combined_intent = LogisticRegression(max_iter=1000, class_weight='balanced')
model_combined_intent.fit(X_train_c_tfidf, y_train_c)

y_pred_c = model_combined_intent.predict(X_test_c_tfidf)
print(classification_report(y_test_c, y_pred_c))

              precision    recall  f1-score   support

           0       0.74      0.91      0.82        32
           1       1.00      0.91      0.95        32
           2       0.91      0.91      0.91        22
           3       1.00      0.88      0.94        25
           4       0.75      0.88      0.81        17
           5       1.00      0.92      0.96        25
           6       0.74      0.76      0.75        34
           7       0.83      0.94      0.88        36
           8       0.90      0.84      0.87        31
           9       1.00      0.96      0.98       200
          10       0.91      0.94      0.92        31
          11       1.00      0.96      0.98        26
          12       0.57      0.67      0.62        12
          13       0.84      0.87      0.86        31
          14       0.75      0.55      0.63        22
          15       0.87      0.96      0.92        28
          16       0.68      0.77      0.72        22
          17       0.89    

In [20]:
# Cell: Save combined intent model
joblib.dump(model_combined_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_combined, '../models/intent_vectorizer.pkl')

combined_label_map = {int(i): label for i, label in enumerate(le.classes_)}
with open('../models/intent_label_map.json', 'w') as f:
    json.dump(combined_label_map, f)

print("Saved. Total intents:", len(combined_label_map))

Saved. Total intents: 104


In [21]:
import pandas as pd

sentiment_df = pd.read_csv('../data/sentiment_cleaned.csv')
intent_df = pd.read_csv('../data/intent_cleaned.csv')

print(sentiment_df.shape, intent_df.shape)

(45615, 3) (9993, 4)


In [22]:
from datasets import load_dataset

general_data = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
general_df = general_data['train'].to_pandas()

print(general_df.shape)
print(general_df.columns.tolist())
print(general_df[['instruction', 'intent']].head())

(26872, 5)
['flags', 'instruction', 'category', 'intent', 'response']
                                         instruction        intent
0   question about cancelling order {{Order Number}}  cancel_order
1  i have a question about cancelling oorder {{Or...  cancel_order
2    i need help cancelling puchase {{Order Number}}  cancel_order
3         I need to cancel purchase {{Order Number}}  cancel_order
4  I cannot afford this order, cancel purchase {{...  cancel_order


In [23]:
banking_subset = intent_df[['text', 'label_text']].rename(columns={'label_text': 'intent_label'})

general_subset = general_df[['instruction', 'intent']].rename(columns={'instruction': 'text', 'intent': 'intent_label'})

combined_df = pd.concat([banking_subset, general_subset], ignore_index=True)
combined_df = combined_df.dropna(subset=['text', 'intent_label'])

print("Combined shape:", combined_df.shape)
print("Total unique intents:", combined_df['intent_label'].nunique())
print(combined_df['intent_label'].value_counts().tail(10))

Combined shape: (36865, 2)
Total unique intents: 104
intent_label
top_up_limits                  97
get_disposable_virtual_card    97
receiving_money                95
compromised_card               86
atm_support                    86
lost_or_stolen_card            82
card_swallowed                 61
card_acceptance                59
virtual_card_not_working       41
contactless_not_working        35
Name: count, dtype: int64


In [24]:
import re
import spacy
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text not in stop_words and len(token.text) > 1]
    return ' '.join(tokens)

In [25]:
from tqdm import tqdm
tqdm.pandas()

combined_df['clean_text'] = combined_df['text'].progress_apply(clean_text)

100%|██████████| 36865/36865 [05:29<00:00, 111.99it/s]


In [26]:
combined_df.to_csv('../data/combined_intent_cleaned.csv', index=False)
print("Saved.")

Saved.


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib
import json

le = LabelEncoder()
combined_df['label_encoded'] = le.fit_transform(combined_df['intent_label'])

X_c = combined_df['clean_text']
y_c = combined_df['label_encoded']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

vectorizer_combined = TfidfVectorizer(max_features=8000)
X_train_c_tfidf = vectorizer_combined.fit_transform(X_train_c)
X_test_c_tfidf = vectorizer_combined.transform(X_test_c)

model_combined_intent = LogisticRegression(max_iter=1000, class_weight='balanced')
model_combined_intent.fit(X_train_c_tfidf, y_train_c)

y_pred_c = model_combined_intent.predict(X_test_c_tfidf)
print(classification_report(y_test_c, y_pred_c))

              precision    recall  f1-score   support

           0       0.74      0.91      0.82        32
           1       1.00      0.91      0.95        32
           2       0.91      0.91      0.91        22
           3       1.00      0.88      0.94        25
           4       0.75      0.88      0.81        17
           5       1.00      0.92      0.96        25
           6       0.74      0.76      0.75        34
           7       0.83      0.94      0.88        36
           8       0.90      0.84      0.87        31
           9       1.00      0.96      0.98       200
          10       0.91      0.94      0.92        31
          11       1.00      0.96      0.98        26
          12       0.57      0.67      0.62        12
          13       0.84      0.87      0.86        31
          14       0.75      0.55      0.63        22
          15       0.87      0.96      0.92        28
          16       0.68      0.77      0.72        22
          17       0.89    

In [28]:
joblib.dump(model_combined_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_combined, '../models/intent_vectorizer.pkl')

combined_label_map = {int(i): label for i, label in enumerate(le.classes_)}
with open('../models/intent_label_map.json', 'w') as f:
    json.dump(combined_label_map, f)

print("Saved. Total intents:", len(combined_label_map))

Saved. Total intents: 104


In [29]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'backend'))

# reload pipeline fresh since model files just changed
import importlib
if 'pipeline' in sys.modules:
    importlib.reload(sys.modules['pipeline'])
from pipeline import analyze_feedback

test_sentences = [
    "Can you tell me your working hours?",
    "I want to cancel my order",
    "My card hasn't arrived yet, it's been two weeks.",
    "This product is amazing, exactly what I needed!",
]

for text in test_sentences:
    result = analyze_feedback(text)
    print(f"{text}\n  -> Intent: {result['intent']} (confidence: {result['intent_confidence']:.2f})\n")

Can you tell me your working hours?
  -> Intent: pending_top_up (confidence: 0.04)

I want to cancel my order
  -> Intent: cancel_order (confidence: 0.67)

My card hasn't arrived yet, it's been two weeks.
  -> Intent: card_arrival (confidence: 0.82)

This product is amazing, exactly what I needed!
  -> Intent: place_order (confidence: 0.16)



In [1]:
import random

random.seed(42)

subjects = ["the app", "your service", "the product", "the support team", "my order",
            "the website", "your team", "the experience", "everything", "the whole process"]
positive_openers = ["Absolutely fantastic", "Wow, I'm impressed", "Really happy with",
                     "Genuinely surprised how good", "Pleasantly surprised by", "Loved"]
negative_openers = ["Honestly disappointed with", "Not impressed by", "Frustrated with",
                     "Let down by", "Unhappy with", "Underwhelmed by"]
neutral_openers = ["It was okay,", "Nothing special about", "An average experience with",
                    "Mixed feelings about", "Not sure how I feel about"]
endings = ["exactly as promised.", "better than expected.", "worse than I hoped.",
           "took longer than expected.", "arrived earlier than expected.",
           "just did what it was supposed to.", "left a lot to be desired.",
           "exceeded my expectations.", "fell short of what I wanted.",
           "was a mixed bag overall."]

general_feedback_examples = []
for opener in positive_openers + negative_openers + neutral_openers:
    for ending in endings:
        for subj in random.sample(subjects, 3):
            general_feedback_examples.append(f"{opener} {subj}, {ending}")

general_feedback_examples = list(set(general_feedback_examples))
random.shuffle(general_feedback_examples)
general_feedback_examples = general_feedback_examples[:180]

print("Generated examples:", len(general_feedback_examples))
print(general_feedback_examples[:5])

Generated examples: 180
['An average experience with the product, better than expected.', 'Mixed feelings about your team, exceeded my expectations.', 'Underwhelmed by the product, exactly as promised.', 'Unhappy with everything, took longer than expected.', 'Underwhelmed by your team, was a mixed bag overall.']


In [4]:
import pandas as pd

combined_df = pd.read_csv('../data/combined_intent_cleaned.csv')
print(combined_df.shape)
print(combined_df.columns.tolist())

(36865, 3)
['text', 'intent_label', 'clean_text']


In [6]:
general_feedback_df = pd.DataFrame({
    'text': general_feedback_examples,
    'intent_label': 'general_feedback'
})

combined_df = pd.concat([combined_df[['text', 'intent_label']], general_feedback_df], ignore_index=True)
print("New combined shape:", combined_df.shape)
print("Total unique intents:", combined_df['intent_label'].nunique())

New combined shape: (37225, 2)
Total unique intents: 105


In [8]:
import re
import spacy
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.text not in stop_words and len(token.text) > 1]
    return ' '.join(tokens)

In [9]:
from tqdm import tqdm
tqdm.pandas()

combined_df['clean_text'] = combined_df['text'].progress_apply(clean_text)
combined_df.to_csv('../data/combined_intent_cleaned_v2.csv', index=False)
print("Saved. Final shape:", combined_df.shape)

100%|██████████| 37225/37225 [01:57<00:00, 317.13it/s]


Saved. Final shape: (37225, 3)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib, json

le = LabelEncoder()
combined_df['label_encoded'] = le.fit_transform(combined_df['intent_label'])

X_c = combined_df['clean_text']
y_c = combined_df['label_encoded']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

vectorizer_combined = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))
X_train_c_tfidf = vectorizer_combined.fit_transform(X_train_c)
X_test_c_tfidf = vectorizer_combined.transform(X_test_c)

model_combined_intent = LogisticRegression(max_iter=1500, class_weight='balanced', C=2.0)
model_combined_intent.fit(X_train_c_tfidf, y_train_c)

y_pred_c = model_combined_intent.predict(X_test_c_tfidf)
print(classification_report(y_test_c, y_pred_c))

              precision    recall  f1-score   support

           0       0.91      0.94      0.92        32
           1       1.00      0.94      0.97        32
           2       0.95      0.95      0.95        22
           3       1.00      0.92      0.96        25
           4       0.82      0.82      0.82        17
           5       1.00      0.92      0.96        25
           6       0.89      0.71      0.79        34
           7       0.83      0.97      0.90        36
           8       0.90      0.84      0.87        31
           9       1.00      0.96      0.98       200
          10       0.88      0.94      0.91        31
          11       0.89      0.96      0.93        26
          12       0.67      0.50      0.57        12
          13       0.83      0.81      0.82        31
          14       0.69      0.50      0.58        22
          15       0.93      0.96      0.95        28
          16       0.81      0.77      0.79        22
          17       0.79    

In [11]:
sentiment_df = pd.read_csv('../data/sentiment_cleaned.csv')

X_s = sentiment_df['clean_text']
y_s = sentiment_df['label']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

vectorizer_sent = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))
X_train_s_tfidf = vectorizer_sent.fit_transform(X_train_s)
X_test_s_tfidf = vectorizer_sent.transform(X_test_s)

model_sentiment = LogisticRegression(max_iter=1500, class_weight='balanced', C=2.0)
model_sentiment.fit(X_train_s_tfidf, y_train_s)

y_pred_s = model_sentiment.predict(X_test_s_tfidf)
print(classification_report(y_test_s, y_pred_s, target_names=['negative', 'neutral', 'positive']))

              precision    recall  f1-score   support

    negative       0.45      0.60      0.51      1419
     neutral       0.66      0.61      0.63      4134
    positive       0.69      0.67      0.68      3570

    accuracy                           0.63      9123
   macro avg       0.60      0.62      0.61      9123
weighted avg       0.64      0.63      0.63      9123



In [12]:
joblib.dump(model_combined_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_combined, '../models/intent_vectorizer.pkl')
combined_label_map = {int(i): label for i, label in enumerate(le.classes_)}
with open('../models/intent_label_map.json', 'w') as f:
    json.dump(combined_label_map, f)

joblib.dump(model_sentiment, '../models/sentiment_model.pkl')
joblib.dump(vectorizer_sent, '../models/sentiment_vectorizer.pkl')

print("All models saved.")

All models saved.


In [13]:
import importlib, sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'backend'))
import pipeline
importlib.reload(pipeline)
from pipeline import analyze_feedback

tests = [
    "Absolutely fantastic service — I only had to contact support five times to get my problem fixed.",
    "Wow, I'm actually impressed. The product works exactly as promised and arrived earlier than expected.",
    "Wonderful experience! Nothing makes my day better than paying for something that doesn't work.",
    "I was ready to write a complaint, but then everything worked perfectly.",
    "Can you tell me your working hours?",
]

for t in tests:
    r = analyze_feedback(t)
    print(f"{t}\n  -> sentiment: {r['sentiment']} ({r['sentiment_confidence']})  |  intent: {r['intent']} ({r['intent_confidence']})  needs_review: {r['intent_confidence'] < 0.3}\n")

Absolutely fantastic service — I only had to contact support five times to get my problem fixed.
  -> sentiment: positive (0.6)  |  intent: contact_customer_service (0.094)  needs_review: True

Wow, I'm actually impressed. The product works exactly as promised and arrived earlier than expected.
  -> sentiment: positive (0.414)  |  intent: general_feedback (0.969)  needs_review: False

Wonderful experience! Nothing makes my day better than paying for something that doesn't work.
  -> sentiment: positive (0.545)  |  intent: top_up_failed (0.058)  needs_review: True

I was ready to write a complaint, but then everything worked perfectly.
  -> sentiment: positive (0.6)  |  intent: complaint (0.073)  needs_review: True

Can you tell me your working hours?
  -> sentiment: neutral (0.478)  |  intent: pending_top_up (0.043)  needs_review: True



In [14]:
general_feedback_examples = [
    "Absolutely fantastic service, I only had to contact support five times to get my problem fixed.",
    "Wow, I'm actually impressed. The product works exactly as promised and arrived earlier than expected.",
    "Wonderful experience! Nothing makes my day better than paying for something that doesn't work.",
    "I was ready to write a complaint, but then everything worked perfectly.",
    "Honestly one of the smoothest experiences I've had with a company in a while.",
    "Not going to lie, I expected this to be a disaster, but it actually went fine.",
    "Can't complain, everything happened exactly the way it was supposed to.",
    "This exceeded my expectations by a mile, genuinely impressed.",
    "It was fine I guess, nothing that stood out either way.",
    "Somewhere between okay and good, hard to really rate it.",
    "I've had better, but I've also had way worse, so it's a wash.",
    "What a joke. Three days of back and forth for something that should've taken minutes.",
    "Never again. This was hands down the worst experience I've had with any company.",
    "I'm speechless, and not in a good way.",
    "Sure, take your sweet time responding to a five-star review, why not.",
    "They really outdid themselves this time, and not in the way they'd want.",
    "Ten out of ten, would not recommend to my worst enemy.",
    "The whole thing just felt off from start to finish, hard to explain why.",
    "Genuinely didn't expect to enjoy this as much as I did.",
    "Color me surprised, this actually lived up to the hype.",
    "I walked in with zero expectations and walked out impressed.",
    "Every step of this felt like it was designed to waste my time.",
    "For once, a company that actually delivers what it promises.",
    "I've told everyone I know how good this was.",
    "I've told everyone I know to stay far away from this.",
    "It is what it is, I've seen worse.",
    "Left feeling pretty underwhelmed by the whole thing.",
    "This restored my faith in decent customer service.",
    "This is exactly why I stopped trusting reviews.",
    "A pleasant surprise from start to finish.",
    "An unpleasant surprise from start to finish.",
    "Didn't think it could get worse, then it did.",
    "Didn't think it could get better, then it did.",
    "Just wanted to say thank you, this made my whole week.",
    "Just wanted to vent, this ruined my whole week.",
    "Overall a solid experience, nothing to write home about but nothing bad either.",
    "I'm honestly torn on how to rate this.",
    "This felt like talking to a wall for twenty minutes straight.",
    "Genuinely one of the better experiences I've had this year.",
    "Genuinely one of the worst experiences I've had this year.",
]

general_inquiry_examples = [
    "Can you tell me your working hours?",
    "What time do you open on weekends?",
    "Is customer support available 24/7?",
    "Do you have a physical store I can visit?",
    "What's the best way to reach someone on your team?",
    "Are you open on public holidays?",
    "How long have you been in business?",
    "Do you offer support in other languages?",
    "What's your company's policy on data privacy?",
    "Can I speak to a real person instead of a bot?",
    "Where are you located?",
    "Do you have a phone number I can call?",
    "What time zone is your support team in?",
    "Is there a live chat option available?",
    "How can I get in touch with your team directly?",
    "What are your holiday hours this year?",
    "Do you have an office I can visit in person?",
    "Is there someone available right now to help me?",
]

print(len(general_feedback_examples), len(general_inquiry_examples))

40 18


In [15]:
combined_df = pd.read_csv('../data/combined_intent_cleaned.csv')[['text', 'intent_label']]

new_rows = pd.DataFrame({
    'text': general_feedback_examples + general_inquiry_examples,
    'intent_label': (['general_feedback'] * len(general_feedback_examples)) +
                    (['general_inquiry'] * len(general_inquiry_examples))
})

combined_df = pd.concat([combined_df, new_rows], ignore_index=True)
print("New combined shape:", combined_df.shape)
print("Total unique intents:", combined_df['intent_label'].nunique())

New combined shape: (36923, 2)
Total unique intents: 106


In [16]:
from tqdm import tqdm
tqdm.pandas()
combined_df['clean_text'] = combined_df['text'].progress_apply(clean_text)
combined_df.to_csv('../data/combined_intent_cleaned_v3.csv', index=False)

100%|██████████| 36923/36923 [02:09<00:00, 285.83it/s]


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib, json

le = LabelEncoder()
combined_df['label_encoded'] = le.fit_transform(combined_df['intent_label'])

X_c = combined_df['clean_text']
y_c = combined_df['label_encoded']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

vectorizer_combined = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))
X_train_c_tfidf = vectorizer_combined.fit_transform(X_train_c)
X_test_c_tfidf = vectorizer_combined.transform(X_test_c)

model_combined_intent = LogisticRegression(max_iter=1500, class_weight='balanced', C=2.0)
model_combined_intent.fit(X_train_c_tfidf, y_train_c)

y_pred_c = model_combined_intent.predict(X_test_c_tfidf)
print(classification_report(y_test_c, y_pred_c))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        32
           1       1.00      0.91      0.95        32
           2       1.00      0.95      0.98        22
           3       1.00      0.92      0.96        25
           4       0.79      0.88      0.83        17
           5       1.00      0.92      0.96        25
           6       0.71      0.74      0.72        34
           7       0.87      0.94      0.91        36
           8       0.90      0.84      0.87        31
           9       1.00      0.96      0.98       200
          10       0.91      0.94      0.92        31
          11       0.93      0.96      0.94        26
          12       0.60      0.50      0.55        12
          13       0.83      0.77      0.80        31
          14       0.65      0.59      0.62        22
          15       0.90      1.00      0.95        28
          16       0.77      0.77      0.77        22
          17       0.74    

In [18]:
joblib.dump(model_combined_intent, '../models/intent_model.pkl')
joblib.dump(vectorizer_combined, '../models/intent_vectorizer.pkl')
combined_label_map = {int(i): label for i, label in enumerate(le.classes_)}
with open('../models/intent_label_map.json', 'w') as f:
    json.dump(combined_label_map, f)

import importlib, sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'backend'))
import pipeline
importlib.reload(pipeline)
from pipeline import analyze_feedback

tests = [
    "Absolutely fantastic service — I only had to contact support five times to get my problem fixed.",
    "Wow, I'm actually impressed. The product works exactly as promised and arrived earlier than expected.",
    "Wonderful experience! Nothing makes my day better than paying for something that doesn't work.",
    "I was ready to write a complaint, but then everything worked perfectly.",
    "Can you tell me your working hours?",
]
for t in tests:
    r = analyze_feedback(t)
    print(f"{t}\n  -> intent: {r['intent']} ({r['intent_confidence']})  needs_review: {r['intent_confidence'] < 0.3}\n")

Absolutely fantastic service — I only had to contact support five times to get my problem fixed.
  -> intent: general_feedback (0.72)  needs_review: False

Wow, I'm actually impressed. The product works exactly as promised and arrived earlier than expected.
  -> intent: general_feedback (0.794)  needs_review: False

Wonderful experience! Nothing makes my day better than paying for something that doesn't work.
  -> intent: general_feedback (0.769)  needs_review: False

I was ready to write a complaint, but then everything worked perfectly.
  -> intent: general_feedback (0.734)  needs_review: False

Can you tell me your working hours?
  -> intent: contact_customer_service (0.046)  needs_review: True

